# Dyad Learning Experiment

## Research Question
**Is learning in a dyad beneficial for the learning process of a reinforcement learning agent?**

## Setup
- **Environment**: Netslide puzzle from Simon Tatham's Portable Puzzle Collection (Gymnasium + Pygame + C backend)
- **Puzzle params**: `2x3b1` (small) and `3x3b1` (medium)
- **Algorithm**: DQN with experience replay, target network, and action masking
- **Observations**: Discrete internal game state (MLP agent) or RGB pixels (CNN agent)
- **Config**: Hydra YAML configs under `config/`; experiment presets under `config/experiment/`

## Experiments
| # | Name | Agent | Obs Type | Config |
|---|------|-------|----------|--------|
| 1 | Baseline MLP | MLP DQN | Discrete state | `+experiment=baseline_mlp` |
| 2 | Baseline CNN | CNN DQN | RGB pixels | `+experiment=baseline_cnn` |
| 3 | Dyad Learning | MLP + CNN | Both (shared) | `+experiment=dyad` |

## Outputs
- **Checkpoints**: `checkpoints/{experiment_name}/`
- **Training CSV**: `results/{experiment_name}/{agent}_training.csv`
- **Eval JSON**: `results/{experiment_name}/{agent}_eval.json`
- **Plots**: Generated inline below

## 0. Imports & Setup

In [1]:
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

from evaluation.analyze import (
    load_training_csv,
    load_eval_json,
    smooth,
    plot_learning_curves,
    plot_eval_comparison,
)

# Ensure plots render inline
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

print("Imports OK")

Imports OK


## 1. Environment Sanity Check
Quick verification that the environment loads and responds to actions.

In [ ]:
import gymnasium as gym
import rlp  # registers rlp/Puzzle-v0

env = gym.make(
    "rlp/Puzzle-v0",
    puzzle="netslide",
    render_mode="rgb_array",
    params="2x3b1",
    window_width=128,
    window_height=128,
    obs_type="puzzle_state",
    include_cursor_in_state_info=True,
)

obs, info = env.reset(seed=42)
print(f"Obs space:    {env.observation_space}")
print(f"Action space: {env.action_space} ({env.action_space.n} actions)")
print(f"Obs keys:     {list(obs.keys())}")
print(f"Info keys:    {list(info.keys())}")
print(f"Action mask:  {env.unwrapped.action_masks()}")

# Take a few random steps
for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"  Step {i+1}: action={action}, reward={reward}, done={terminated or truncated}")
    print(f"Observation state: {obs}")
    if terminated or truncated:
        obs, info = env.reset()

env.close()
print("Environment OK")

Obs space:    Dict('barriers': Box(0, 256, (6,), uint16), 'cursor_pos': Box(-1, 3, (2,), int32), 'height': Discrete(3, start=1), 'last_move_col': Discrete(3, start=-1), 'last_move_dir': Discrete(4, start=-1), 'last_move_row': Discrete(4, start=-1), 'move_count': Box(0, 2147483647, (1,), int32), 'movetarget': Box(0, 2147483647, (1,), int32), 'tiles': Box(0, 31, (6,), uint16), 'width': Discrete(2, start=1), 'wrapping': Discrete(2))
Action space: Discrete(5) (5 actions)
Obs keys:     ['width', 'height', 'wrapping', 'move_count', 'movetarget', 'last_move_row', 'last_move_col', 'last_move_dir', 'tiles', 'barriers', 'cursor_pos']
Info keys:    ['puzzle_state', 'state_histogram', 'current_state_repeats', 'current_move_was_toward_solution', 'current_move_was_cursor_move']
Action mask:  [0 1 1 1 1]
  Step 1: action=3, reward=0, done=False


KeyError: 'pixels'

---
## 2. Training Helper
Utility function to launch training as a subprocess (avoids Hydra re-initialization issues in notebooks).

In [ ]:
def run_experiment(hydra_args: list[str], timeout: int | None = None) -> int:
    """Launch experiment.py with the given Hydra overrides as a subprocess.

    Args:
        hydra_args: List of Hydra CLI arguments (e.g. ['+experiment=baseline_mlp']).
        timeout: Optional timeout in seconds.

    Returns:
        Subprocess return code (0 = success).
    """
    cmd = [sys.executable, "experiment.py"] + hydra_args
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, timeout=timeout, capture_output=False)
    print(f"Exit code: {result.returncode}")
    return result.returncode

: 

---
## 3. Experiment 1 — Baseline MLP (Discrete State)
Train a single DQN agent with MLP network on the discrete internal game state.

**Netslide 2x3b1:**

In [ ]:
run_experiment([
    "+experiment=baseline_mlp",
    "experiment_name=exp1_mlp_2x3",
    "env=netslide_2x3",
])

Running: /home/f133r/projects/dyad_rl/.venv/bin/python experiment.py +experiment=baseline_mlp experiment_name=exp1_mlp_2x3 env=netslide_2x3


(null): No such file or directory
(null): No such file or directory


[2026-03-02 18:37:58,510][__main__][INFO] - Device: cuda
[2026-03-02 18:37:58,510][__main__][INFO] - Config:
env:
  puzzle: netslide
  params: 2x3b1
  obs_type: puzzle_state
  render_mode: rgb_array
  window_width: 128
  window_height: 128
  allow_undo: false
  max_state_repeats: 200
  include_cursor_in_state_info: true
agent:
  type: mlp
  hidden_sizes:
  - 64
  - 64
  batch_size: 32
  gamma: 0.99
  eps_start: 0.9
  eps_end: 0.05
  eps_decay: 100000
  tau: 0.1
  lr: 0.0001
  buffer_size: 50000
training:
  total_episodes: 10000
  max_steps: 5000
  eval_interval: 1000
  eval_episodes: 100
  checkpoint_interval: 5000
  log_interval: 50
  prefill_buffer: true
  dyad: false
mode: train
experiment_name: exp1_mlp_2x3
seed: 42
device: auto

[2026-03-02 18:37:58,520][__main__][INFO] - Obs shape: (34,), Actions: 5
[2026-03-02 18:37:58,943][training.train_single][INFO] - Prefilling replay buffer with 50000 random transitions...
[2026-03-02 18:38:07,232][training.train_single][INFO] - Prefilled b

**Netslide 3x3b1:**

In [ ]:
run_experiment([
    "+experiment=baseline_mlp",
    "experiment_name=exp1_mlp_3x3",
    "env=netslide_3x3",
])

---
## 4. Experiment 2 — Baseline CNN (RGB Pixels)
Train a single DQN agent with CNN network on RGB pixel observations.

**Netslide 2x3b1:**

In [ ]:
run_experiment([
    "+experiment=baseline_cnn",
    "experiment_name=exp2_cnn_2x3",
    "env=netslide_2x3",
])

**Netslide 3x3b1:**

In [ ]:
run_experiment([
    "+experiment=baseline_cnn",
    "experiment_name=exp2_cnn_3x3",
    "env=netslide_3x3",
])

---
## 5. Experiment 3 — Dyad Learning
Train two agents (MLP on discrete state + CNN on RGB) simultaneously with periodic experience sharing.
Each agent rates the other's trajectories using its own value function and accepts transitions above the rating threshold.

**Netslide 2x3b1:**

In [ ]:
run_experiment([
    "+experiment=dyad",
    "experiment_name=exp3_dyad_2x3",
    "env=netslide_2x3",
])

**Netslide 3x3b1:**

In [ ]:
run_experiment([
    "+experiment=dyad",
    "experiment_name=exp3_dyad_3x3",
    "env=netslide_3x3",
])

---
## Hyperparameter Optimization (Optuna)
Use Hydra-Optuna sweeper to search for optimal hyperparameters before running full experiments.
Three sweep configs are provided: `sweep_mlp`, `sweep_cnn`, `sweep_dyad`.

Each sweep maximizes the best evaluation win rate. By default, 50 trials using TPE sampler.
Adjust `hydra.sweeper.n_trials` to control the number of trials.

**Sweep MLP:**

In [ ]:
run_experiment([
    "--multirun",
    "--config-name=sweep_mlp",
])

**Sweep CNN:**

In [ ]:
run_experiment([
    "--multirun",
    "--config-name=sweep_cnn",
])

**Sweep Dyad:**

In [ ]:
run_experiment([
    "--multirun",
    "--config-name=sweep_dyad",
])

---
## 6. Results Analysis

### 6.1 Learning Curves — Netslide 2x3b1
Compare training return and win rate across all three experiments on the smaller puzzle.

In [ ]:
# --- 2x3b1 Learning Curves ---
training_csvs_2x3 = {
    "Exp1 MLP": "results/exp1_mlp_2x3/exp1_mlp_2x3_training.csv",
    "Exp2 CNN": "results/exp2_cnn_2x3/exp2_cnn_2x3_training.csv",
    "Exp3 Dyad (Agent A / MLP)": "results/exp3_dyad_2x3/agent_a_training.csv",
    "Exp3 Dyad (Agent B / CNN)": "results/exp3_dyad_2x3/agent_b_training.csv",
}

# Filter to existing files only
available = {k: v for k, v in training_csvs_2x3.items() if os.path.exists(v)}
if available:
    plot_learning_curves(available, "results/visualizations/2x3", window=100)

    # Also show inline
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, csv_path in available.items():
        data = load_training_csv(csv_path)
        eps = data["episode"]
        ret_s = smooth(data["total_return"], 100)
        axes[0].plot(eps[99:], ret_s, label=name, alpha=0.8)
        wr_s = smooth([1.0 if s else 0.0 for s in data["success"]], 100)
        axes[1].plot(eps[99:], wr_s, label=name, alpha=0.8)

    axes[0].set(xlabel="Episode", ylabel="Avg Return", title="Training Return (2x3b1)")
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set(xlabel="Episode", ylabel="Win Rate", title="Training Win Rate (2x3b1)")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No 2x3b1 training results found yet. Run experiments first.")

### 6.2 Learning Curves — Netslide 3x3b1

In [ ]:
# --- 3x3b1 Learning Curves ---
training_csvs_3x3 = {
    "Exp1 MLP": "results/exp1_mlp_3x3/exp1_mlp_3x3_training.csv",
    "Exp2 CNN": "results/exp2_cnn_3x3/exp2_cnn_3x3_training.csv",
    "Exp3 Dyad (Agent A / MLP)": "results/exp3_dyad_3x3/agent_a_training.csv",
    "Exp3 Dyad (Agent B / CNN)": "results/exp3_dyad_3x3/agent_b_training.csv",
}

available = {k: v for k, v in training_csvs_3x3.items() if os.path.exists(v)}
if available:
    plot_learning_curves(available, "results/visualizations/3x3", window=100)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, csv_path in available.items():
        data = load_training_csv(csv_path)
        eps = data["episode"]
        ret_s = smooth(data["total_return"], 100)
        axes[0].plot(eps[99:], ret_s, label=name, alpha=0.8)
        wr_s = smooth([1.0 if s else 0.0 for s in data["success"]], 100)
        axes[1].plot(eps[99:], wr_s, label=name, alpha=0.8)

    axes[0].set(xlabel="Episode", ylabel="Avg Return", title="Training Return (3x3b1)")
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set(xlabel="Episode", ylabel="Win Rate", title="Training Win Rate (3x3b1)")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No 3x3b1 training results found yet. Run experiments first.")

### 6.3 Evaluation Win Rate Comparison
Compare periodic evaluation win rates across experiments.

In [ ]:
# --- Evaluation Win-Rate Comparison (2x3b1) ---
eval_jsons_2x3 = {
    "Exp1 MLP": "results/exp1_mlp_2x3/exp1_mlp_2x3_eval.json",
    "Exp2 CNN": "results/exp2_cnn_2x3/exp2_cnn_2x3_eval.json",
    "Exp3 Dyad (A)": "results/exp3_dyad_2x3/agent_a_eval.json",
    "Exp3 Dyad (B)": "results/exp3_dyad_2x3/agent_b_eval.json",
}
avail_eval = {k: v for k, v in eval_jsons_2x3.items() if os.path.exists(v)}
if avail_eval:
    plot_eval_comparison(avail_eval, "results/visualizations/2x3")

    fig, ax = plt.subplots(figsize=(8, 5))
    for name, json_path in avail_eval.items():
        evals = load_eval_json(json_path)
        ax.plot([e["episode"] for e in evals], [e["win_rate"] for e in evals],
                marker="o", label=name, alpha=0.8)
    ax.set(xlabel="Episode", ylabel="Win Rate", title="Eval Win Rate (2x3b1)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No 2x3b1 eval results found yet.")

# --- Evaluation Win-Rate Comparison (3x3b1) ---
eval_jsons_3x3 = {
    "Exp1 MLP": "results/exp1_mlp_3x3/exp1_mlp_3x3_eval.json",
    "Exp2 CNN": "results/exp2_cnn_3x3/exp2_cnn_3x3_eval.json",
    "Exp3 Dyad (A)": "results/exp3_dyad_3x3/agent_a_eval.json",
    "Exp3 Dyad (B)": "results/exp3_dyad_3x3/agent_b_eval.json",
}
avail_eval_3 = {k: v for k, v in eval_jsons_3x3.items() if os.path.exists(v)}
if avail_eval_3:
    plot_eval_comparison(avail_eval_3, "results/visualizations/3x3")

    fig, ax = plt.subplots(figsize=(8, 5))
    for name, json_path in avail_eval_3.items():
        evals = load_eval_json(json_path)
        ax.plot([e["episode"] for e in evals], [e["win_rate"] for e in evals],
                marker="o", label=name, alpha=0.8)
    ax.set(xlabel="Episode", ylabel="Win Rate", title="Eval Win Rate (3x3b1)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No 3x3b1 eval results found yet.")

### 6.4 Final Summary Table
Aggregate best evaluation win rates from all experiments.

In [ ]:
all_eval_files = {
    # 2x3b1
    ("2x3b1", "Exp1 MLP"): "results/exp1_mlp_2x3/exp1_mlp_2x3_eval.json",
    ("2x3b1", "Exp2 CNN"): "results/exp2_cnn_2x3/exp2_cnn_2x3_eval.json",
    ("2x3b1", "Exp3 Dyad A"): "results/exp3_dyad_2x3/agent_a_eval.json",
    ("2x3b1", "Exp3 Dyad B"): "results/exp3_dyad_2x3/agent_b_eval.json",
    # 3x3b1
    ("3x3b1", "Exp1 MLP"): "results/exp1_mlp_3x3/exp1_mlp_3x3_eval.json",
    ("3x3b1", "Exp2 CNN"): "results/exp2_cnn_3x3/exp2_cnn_3x3_eval.json",
    ("3x3b1", "Exp3 Dyad A"): "results/exp3_dyad_3x3/agent_a_eval.json",
    ("3x3b1", "Exp3 Dyad B"): "results/exp3_dyad_3x3/agent_b_eval.json",
}

print(f"{'Puzzle':<8} {'Experiment':<18} {'Best WR':>8} {'Best Ret':>9} {'@ Episode':>10}")
print("-" * 58)
for (puzzle, exp_name), path in all_eval_files.items():
    if not os.path.exists(path):
        continue
    evals = load_eval_json(path)
    if not evals:
        continue
    best = max(evals, key=lambda e: e["win_rate"])
    print(f"{puzzle:<8} {exp_name:<18} {best['win_rate']:>8.3f} {best['avg_return']:>9.1f} {best['episode']:>10}")